In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from mpstemmer import MPStemmer

import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from pprint import pprint

from transformers import BertTokenizer
from nltk.tokenize import RegexpTokenizer

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sklearn.feature_extraction.text import CountVectorizer

# os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("Total GPU:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

2024-08-18 21:44:02.642955: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Total GPU: 1
Current GPU: NVIDIA RTX A5000


In [2]:
path = "../bert_data/id-p2"

train_files = []
val_files = []
test_files = []
for file in os.listdir(path):
    if "bert.pt" in file and "train" in file:
        train_files.append(path + "/" + file)
    elif "bert.pt" in file and "valid" in file:
        val_files.append(path + "/" + file)
    elif "bert.pt" in file and "test" in file:
        test_files.append(path + "/" + file)

train_files = sorted(train_files)
val_files = sorted(val_files)
test_files = sorted(test_files)

In [3]:
train_files = train_files[:]
val_files = val_files[:]
test_files = test_files[:]

In [4]:
train_docs = []

i = 0
for file in train_files:
    print(f"Loading data train {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        train_docs.append(src)
    i = i + 1

Loading data train 0...
Loading data train 1...
Loading data train 2...
Loading data train 3...
Loading data train 4...
Loading data train 5...
Loading data train 6...
Loading data train 7...
Loading data train 8...
Loading data train 9...
Loading data train 10...
Loading data train 11...
Loading data train 12...
Loading data train 13...
Loading data train 14...
Loading data train 15...
Loading data train 16...
Loading data train 17...
Loading data train 18...
Loading data train 19...


In [5]:
val_docs = []

i = 0
for file in val_files:
    print(f"Loading data val {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        val_docs.append(src)
    i = i + 1

Loading data val 0...
Loading data val 1...
Loading data val 2...


In [6]:
test_docs = []

i = 0
for file in test_files:
    print(f"Loading data test {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        test_docs.append(src)
    i = i + 1

Loading data test 0...
Loading data test 1...
Loading data test 2...


In [7]:
# Remove stop words
def remove_stop_words(doc):
    factory = StopWordRemoverFactory()
    stopword = factory.create_stop_word_remover()
    res = stopword.remove(doc)
    return res

In [8]:
rm_train_docs = []
rm_val_docs = []
rm_test_docs = []

# Preprocess only removing stop words
# If we also use stemming, the topic will be non-sense
for doc in train_docs:
    # print("Preprocess train docs...")
    doc = remove_stop_words(doc)
    rm_train_docs.append(doc)

for doc in val_docs:
    # print("Preprocess val docs...")
    doc = remove_stop_words(doc)
    rm_val_docs.append(doc)

for doc in test_docs:
    # print("Preprocess test docs...")
    doc = remove_stop_words(doc)
    rm_test_docs.append(doc)

In [9]:
print("Saving docs...")
torch.save(rm_train_docs, "./data/rm_train_doc.pt")
torch.save(rm_val_docs, "./data/rm_val_doc.pt")
torch.save(rm_test_docs, "./data/rm_test_doc.pt")

Saving docs...


In [10]:
def tokenize(docs):
    # Split the documents into tokens.
    tokenizer = RegexpTokenizer(r'\w+')
    new_docs = docs.copy()
    for idx in range(len(docs)):
        new_docs[idx] = docs[idx].lower()  # Convert to lowercase.
        new_docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words.
        
    return new_docs

In [11]:
tk_train_docs = tokenize(rm_train_docs)
tk_val_docs = tokenize(rm_train_docs)
tk_test_docs = tokenize(rm_train_docs)

In [12]:
from gensim.models import CoherenceModel

In [13]:
def cal_coh(sbert, n_comps, cluster_size, filename):
    TOP_N_WORDS = 10

    path = f"./topics/{filename}.pt"
    
    # Calculate embeddings
    if sbert == "en":
        embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    else:
        embedding_model = SentenceTransformer("denaya/indoSBERT-large")
    embeddings = embedding_model.encode(rm_train_docs, show_progress_bar=False) 
    
    # Reduce dimension
    umap_model = UMAP(n_neighbors=15, n_components=n_comps, min_dist=0.0, metric='cosine', random_state=42)
    # Clustering
    hdbscan_model = HDBSCAN(min_cluster_size=cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    # Vectorizer
    vectorizer_model = CountVectorizer(ngram_range=(1, 2))
    # Representation model
    representation_model = MaximalMarginalRelevance(diversity=0.2)
    # Model
    topic_model = BERTopic(
      # Pipeline models
      embedding_model=embedding_model,
      umap_model=umap_model,
      hdbscan_model=hdbscan_model,
      vectorizer_model=vectorizer_model,
      representation_model=representation_model,
      # Hyperparameters
      top_n_words=TOP_N_WORDS,
      verbose=False
    )
    
    topics, probs = topic_model.fit_transform(rm_train_docs, embeddings)
    
    # Get the topics and their corresponding words
    topics = topic_model.get_topics()
    torch.save(topics, path)
    
    # Prepare the list of topic words
    topic_words = []
    for topic_num, words in topics.items():
        topic_words.append([word for word, _ in words])
    
    # Preprocess the documents (tokenize, etc.)
    texts = [doc.split() for doc in rm_train_docs]
    
    # Create a dictionary and a corpus
    dictionary = corpora.Dictionary(texts)
    # Filter out words that occur less than 20 documents, or more than 50% of the documents.
    # dictionary.filter_extremes(no_below=20, no_above=0.5)
    
    corpus = [dictionary.doc2bow(text) for text in texts]
    
    # Calculate Topic Diversity
    top_words_set = set()
    total_words = 0
    for topic_num, words in topics.items():
        total_words += len(words)
        for word, _ in words:
            top_words_set.add(word)
    
    num_unique_words = len(top_words_set)
    topic_diversity = num_unique_words / total_words
    
    
    # Calculate Coherence Score using the 'c_v' metric
    coherence_model = CoherenceModel(topics=topic_words, texts=tk_train_docs, dictionary=dictionary, coherence='c_v')
    coherence_score = coherence_model.get_coherence()

    print(f"Statistics {filename} --------------------------")
    print("Total topics:", len(topics))
    print("TD:", topic_diversity)
    print("TC:", coherence_score)
    print()

In [14]:
cal_coh("id", 50, 30, "id-5")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-5 --------------------------
Total topics: 12
TD: 0.8666666666666667
TC: 0.7527923030477096



In [ ]:
cal_coh("en", 5, 20, "en-1")
cal_coh("en", 5, 30, "en-2")
cal_coh("en", 5, 50, "en-3")
cal_coh("en", 50, 20, "en-4")
cal_coh("en", 50, 30, "en-5")
cal_coh("en", 50, 50, "en-6")
cal_coh("en", 100, 20, "en-7")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics en-1 --------------------------
Total topics: 195
TD: 0.8
TC: 0.7384467455853907



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics en-2 --------------------------
Total topics: 142
TD: 0.7992957746478874
TC: 0.7263249170087259



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics en-6 --------------------------
Total topics: 84
TD: 0.7845238095238095
TC: 0.694129508572391



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics en-7 --------------------------
Total topics: 194
TD: 0.7969072164948454
TC: 0.7335145115562091



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [14]:
cal_coh("en", 100, 30, "en-8")
cal_coh("en", 100, 50, "en-9")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics en-8 --------------------------
Total topics: 140
TD: 0.8078571428571428
TC: 0.7189038551976555



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics en-9 --------------------------
Total topics: 87
TD: 0.7804597701149425
TC: 0.705830799792233



In [15]:
cal_coh("id", 5, 20, "id-1")
cal_coh("id", 5, 30, "id-2")
cal_coh("id", 5, 50, "id-3")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-1 --------------------------
Total topics: 157
TD: 0.8656050955414013
TC: 0.7231297781931876



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-2 --------------------------
Total topics: 12
TD: 0.925
TC: 0.7489351525314426



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-3 --------------------------
Total topics: 11
TD: 0.8545454545454545
TC: 0.7616240394261683



In [16]:
cal_coh("id", 50, 20, "id-4")
cal_coh("id", 50, 30, "id-5")
cal_coh("id", 50, 50, "id-6")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-4 --------------------------
Total topics: 144
TD: 0.8666666666666667
TC: 0.7257705714463732



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-6 --------------------------
Total topics: 10
TD: 0.89
TC: 0.7493306927305416



In [18]:
cal_coh("id", 5, 21, "id-7")
cal_coh("id", 100, 23, "id-8")
cal_coh("id", 100, 25, "id-9")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-7 --------------------------
Total topics: 163
TD: 0.8484662576687116
TC: 0.7281279758669438



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-8 --------------------------
Total topics: 130
TD: 0.8707692307692307
TC: 0.7251607145285737



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Statistics id-9 --------------------------
Total topics: 123
TD: 0.883739837398374
TC: 0.7119115187050997

